# Figure 3: Evidence For Reference-Conditioned Modeling

- Figure / panels: `Fig3a` to `Fig3f`, with `Fig3g` exported as an optional reserve panel
- Inputs: resolved metrics files from `artifacts/results/<dataset>/...`, Systema baseline outputs, payload `pkl`
- Outputs: `artifacts/paper_figures/main/Fig3_ReferenceConditioning/*`
- Run order: run after the Fig2 benchmark notebook; this notebook focuses on nearest vs random and reference baselines
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Main text


In [1]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel, stratified_benchmark, systema_mechanism

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)
importlib.reload(stratified_benchmark)
importlib.reload(systema_mechanism)



<module 'scripts.trishift.analysis.systema_mechanism' from 'E:\\CODE\\trishift\\scripts\\trishift\\analysis\\systema_mechanism.py'>

In [2]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'subgroup_filter': None},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'subgroup_filter': None},
    {'dataset_key': 'norman', 'dataset_label': 'Norman (single)', 'subgroup_filter': 'single'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562', 'subgroup_filter': None},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1', 'subgroup_filter': None},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

REFERENCE_MODELS = ['trishift_nearest', 'trishift_random', 'systema_nonctl_mean', 'systema_matching_mean']  # edit here
DISTANCE_MODELS = ['trishift_nearest', 'trishift_random']  # edit here
GENERIC_SHIFT_MODELS = ['trishift_nearest', 'systema_nonctl_mean', 'systema_matching_mean']
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig3_ReferenceConditioning'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_CASE_OVERRIDE = {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}  # set to None for auto
MODEL_LABELS = {
    'trishift_nearest': 'TriShift',
    'trishift_random': 'TriShift random',
    'systema_nonctl_mean': 'Perturbed mean',
    'systema_matching_mean': 'Matching mean',
}
MODEL_COLORS = {
    'TriShift': '#9FD9D3',
    'TriShift random': '#6F8FB8',
    'Perturbed mean': '#B9AEDC',
    'Matching mean': '#8D7BC8',
    'Truth': '#7F7F7F',
}
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Reference case override:', REFERENCE_CASE_OVERRIDE)



Selected datasets: ['adamson', 'norman']
Selected splits: [1, 2, 3, 4, 5]
Reference case override: {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}


In [3]:
def safe_reference_benchmark(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'reference_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        return {'status': 'ready', 'result': baseline_panel.run_baseline_panel(dataset=dataset_key, models=REFERENCE_MODELS, split_ids=SPLIT_IDS, subgroup_filter=subgroup_filter, out_root=out_dir)}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def safe_mechanism_summary(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'mechanism_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        return {
            'status': 'ready',
            'result': systema_mechanism.run_systema_mechanism_analysis(
                dataset=dataset_key,
                models=REFERENCE_MODELS,
                split_ids=SPLIT_IDS,
                subgroup_filter=subgroup_filter,
                out_root=out_dir,
                paths_path=repo_root / 'configs' / 'paths.yaml',
            ),
        }
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def _shrink_bars(ax: plt.Axes, scale: float = 0.9) -> None:
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


def render_grouped_bar(df: pd.DataFrame, metric_col: str, metric_label: str, out_path: Path, order: list[str] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(9.5, 4.7), dpi=220)
    if df.empty:
        ax.text(0.5, 0.5, f'No rows for {metric_col}', ha='center', va='center'); ax.axis('off')
    else:
        work = df.copy()
        if order is not None and 'dataset_label' in work.columns:
            work['dataset_label'] = pd.Categorical(work['dataset_label'], categories=order, ordered=True)
            work = work.sort_values('dataset_label')
        sns.barplot(data=work, x='dataset_label', y=metric_col, hue='label', palette=MODEL_COLORS, ax=ax, saturation=0.95, errorbar=None)
        _shrink_bars(ax, scale=0.9)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.tick_params(axis='x', rotation=32)
        ax.legend(title='', frameon=False, ncol=min(4, work['label'].nunique()), loc='upper left', bbox_to_anchor=(0.02, 1.02), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
        for patch in ax.patches:
            patch.set_edgecolor('black'); patch.set_linewidth(0.5)
        _style_axis(ax)
    fig.tight_layout(); fig.savefig(out_path, bbox_inches='tight'); plt.close(fig)


def _build_reference_case(dataset_key: str, condition: str, split_id: int):
    condition_key = normalize_condition(condition)
    try:
        _, near_payload = load_payload_item(dataset=dataset_key, model_name='trishift_nearest', split_id=split_id, condition=None)
        _, rand_payload = load_payload_item(dataset=dataset_key, model_name='trishift_random', split_id=split_id, condition=None)
    except Exception:
        return None, None
    near_item = {normalize_condition(k): v for k, v in near_payload.items()}.get(condition_key)
    rand_item = {normalize_condition(k): v for k, v in rand_payload.items()}.get(condition_key)
    if near_item is None or rand_item is None:
        return None, None
    truth = np.asarray(near_item['Truth'], dtype=float)
    ctrl = np.asarray(near_item['Ctrl'], dtype=float)
    near_pred = np.asarray(near_item['Pred'], dtype=float)
    rand_pred = np.asarray(rand_item['Pred'], dtype=float)
    genes = np.asarray(near_item.get('DE_name', [f'g{i}' for i in range(near_pred.shape[1])])).astype(str)
    truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    order = np.argsort(-np.abs(truth_delta))[:12]
    rows = [
        pd.DataFrame({'Gene': genes[order], 'Expression': truth_delta[order], 'Group': 'Truth'}),
        pd.DataFrame({'Gene': genes[order], 'Expression': near_pred.mean(axis=0)[order] - ctrl.mean(axis=0)[order], 'Group': 'TriShift'}),
        pd.DataFrame({'Gene': genes[order], 'Expression': rand_pred.mean(axis=0)[order] - ctrl.mean(axis=0)[order], 'Group': 'TriShift random'}),
    ]
    return {'dataset_key': dataset_key, 'condition': condition_key, 'split_id': int(split_id)}, pd.concat(rows, ignore_index=True)


def pick_reference_case():
    if REFERENCE_CASE_OVERRIDE is not None:
        case_meta, case_df = _build_reference_case(
            dataset_key=REFERENCE_CASE_OVERRIDE['dataset_key'],
            condition=REFERENCE_CASE_OVERRIDE['condition'],
            split_id=int(REFERENCE_CASE_OVERRIDE['split_id']),
        )
        if case_meta is not None and case_df is not None:
            case_meta['score_gain'] = np.nan
            case_meta['override'] = True
            return case_meta, case_df
    preferred_order = ['adamson', 'norman']
    for dataset_key in preferred_order:
        try:
            nearest_df = baseline_panel.load_metrics_df(baseline_panel.resolve_result(dataset=dataset_key, model_name='trishift_nearest'))
            random_df = baseline_panel.load_metrics_df(baseline_panel.resolve_result(dataset=dataset_key, model_name='trishift_random'))
        except Exception:
            continue
        if dataset_key == 'norman' and 'subgroup' in nearest_df.columns and 'subgroup' in random_df.columns:
            nearest_df = nearest_df[nearest_df['subgroup'].astype(str) == 'single'].copy()
            random_df = random_df[random_df['subgroup'].astype(str) == 'single'].copy()
        metric_cols = ['condition', 'split_id', 'pearson', 'deg_mean_r2', 'systema_corr_20de_allpert']
        common = nearest_df[metric_cols].merge(random_df[metric_cols], on=['condition', 'split_id'], suffixes=('_nearest', '_random'))
        if common.empty:
            continue
        common['score_gain'] = (common['pearson_nearest'] - common['pearson_random']).fillna(0.0) + (common['deg_mean_r2_nearest'] - common['deg_mean_r2_random']).fillna(0.0) + (common['systema_corr_20de_allpert_nearest'] - common['systema_corr_20de_allpert_random']).fillna(0.0)
        best = common.sort_values('score_gain', ascending=False).iloc[0]
        case_meta, case_df = _build_reference_case(dataset_key=dataset_key, condition=best['condition'], split_id=int(best['split_id']))
        if case_meta is None or case_df is None:
            continue
        case_meta['score_gain'] = float(best['score_gain'])
        case_meta['override'] = False
        return case_meta, case_df
    return None, None



In [4]:
reference_runs, mechanism_runs = [], []
for spec in DATASET_SPECS:
    reference_runs.append({**spec, **safe_reference_benchmark(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})
    mechanism_runs.append({**spec, **safe_mechanism_summary(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})

reference_frames = []
for row in reference_runs:
    if row['status'] == 'ready':
        df = row['result']['summary_df'].copy()
        df['dataset_label'] = row['dataset_label']
        reference_frames.append(df)
reference_summary_df = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame()
reference_summary_df.to_csv(OUT_ROOT / 'reference_summary_all.csv', index=False, encoding='utf-8-sig')

residual_frames, centroid_frames, bin_frames = [], [], []
for row in mechanism_runs:
    if row['status'] != 'ready':
        continue
    spec = row['dataset_label']
    residual_df = row['result']['residual_summary_df'].copy()
    centroid_df = row['result']['centroid_summary_df'].copy()
    bin_df = row['result']['difficulty_bin_df'].copy()
    if not residual_df.empty:
        residual_df['dataset_label'] = row['dataset_label']
        residual_frames.append(residual_df)
    if not centroid_df.empty:
        centroid_df['dataset_label'] = row['dataset_label']
        centroid_frames.append(centroid_df)
    if not bin_df.empty:
        bin_df['dataset_label'] = row['dataset_label']
        bin_frames.append(bin_df)
residual_summary_df = pd.concat(residual_frames, ignore_index=True) if residual_frames else pd.DataFrame()
centroid_summary_df = pd.concat(centroid_frames, ignore_index=True) if centroid_frames else pd.DataFrame()
difficulty_bin_df = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
residual_summary_df.to_csv(OUT_ROOT / 'residualized_systema_summary_all.csv', index=False, encoding='utf-8-sig')
centroid_summary_df.to_csv(OUT_ROOT / 'centroid_accuracy_summary_all.csv', index=False, encoding='utf-8-sig')
difficulty_bin_df.to_csv(OUT_ROOT / 'difficulty_bin_generic_shift_summary_all.csv', index=False, encoding='utf-8-sig')
display(reference_summary_df.head())
display(residual_summary_df.head())
display(centroid_summary_df.head())



[embd] embedding index ready: key=embd_index
[degs] computing with scanpy


D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:440: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:440: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals"] = pvals[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:450: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calli

D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:440: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

[degs] scanpy degs ready
[degs] using precomputed degs: key=top20_degs_non_dropout


...

D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
D:\conda_envs\biolord\lib\site-packages\scanpy\tools\_rank_genes_groups.py:440: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

[degs] scanpy degs ready
[degs] using precomputed degs: key=top20_degs_non_dropout


,dataset,model_name,label,n_rows,split_ids_used,metrics_path,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum,dataset_label
0,adamson,trishift_nearest,TriShift nearest,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\adamson\met...,0.944659,0.168323,0.805440,0.526377,-0.040505,0.945778,0.825881,3.821235,Adamson
1,adamson,trishift_random,TriShift random,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\adamson\met...,0.911021,0.213522,0.758794,0.341554,-0.119977,0.931520,0.759276,4.244670,Adamson
2,adamson,systema_nonctl_mean,Systema nonctl-mean,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\adamson\sys...,0.897379,0.238245,0.733948,0.339018,-0.107215,NaN,NaN,NaN,Adamson
3,adamson,systema_matching_mean,Systema matching-mean,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\adamson\sys...,0.897379,0.238245,0.733948,0.339018,-0.107215,NaN,NaN,NaN,Adamson
4,norman,trishift_nearest,TriShift nearest,105,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\norman\best...,0.758777,0.403461,0.472745,0.585030,0.215709,0.953619,0.571672,5.442953,Norman (single)


,dataset,model_name,systema_corr_20de_allpert,residualized_systema_corr_20de_allpert,generic_projection_ratio,dataset_label
0,adamson,systema_matching_mean,0.339018,0.212542,0.999306,Adamson
1,adamson,systema_nonctl_mean,0.339018,0.212542,0.999306,Adamson
2,adamson,trishift_nearest,0.526377,0.422969,0.813074,Adamson
3,adamson,trishift_random,0.341554,0.385857,0.833967,Adamson
4,norman,systema_matching_mean,0.626297,0.716289,0.681702,Norman (single)


,dataset,model_name,centroid_accuracy,centroid_top1,centroid_top3,centroid_mean_rank,n_conditions,dataset_label
0,adamson,systema_matching_mean,0.500000,0.058824,0.176471,9.000000,17.0,Adamson
1,adamson,systema_nonctl_mean,0.500000,0.058824,0.176471,9.000000,17.0,Adamson
2,adamson,trishift_nearest,0.602206,0.094118,0.258824,7.364706,17.0,Adamson
3,adamson,trishift_random,0.557353,0.105882,0.211765,8.082353,17.0,Adamson
4,norman,systema_matching_mean,0.810952,0.657143,0.695238,4.780952,21.0,Norman (single)


In [5]:
dataset_order = [spec['dataset_label'] for spec in DATASET_SPECS]
nearest_random_df = reference_summary_df[reference_summary_df['model_name'].isin(['trishift_nearest', 'trishift_random'])].copy()
if not nearest_random_df.empty:
    nearest_random_df['label'] = nearest_random_df['model_name'].map({'trishift_nearest': 'TriShift', 'trishift_random': 'TriShift random'})
render_grouped_bar(nearest_random_df, 'mean_pearson', 'Nearest vs random reference retrieval', OUT_ROOT / 'fig3a_nearest_vs_random.png', order=dataset_order)

systema_plot_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
if not systema_plot_df.empty:
    systema_plot_df['label'] = systema_plot_df['model_name'].map(MODEL_LABELS).fillna(systema_plot_df['model_name'])
render_grouped_bar(systema_plot_df, 'mean_systema_corr_20de_allpert', 'Systema Pearson', OUT_ROOT / 'fig3b_systema_pearson.png', order=dataset_order)

signal_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
metric_cols = ['mean_pearson', 'mean_nmse']
if not signal_df.empty:
    signal_df['label'] = signal_df['model_name'].map(MODEL_LABELS).fillna(signal_df['model_name'])
    ref_heatmap = signal_df[['label', 'dataset_label'] + [c for c in metric_cols if c in signal_df.columns]].copy().melt(id_vars=['label', 'dataset_label'], var_name='metric', value_name='value')
    ref_heatmap['metric'] = ref_heatmap['metric'].str.replace('mean_', '', regex=False)
    ref_heatmap['column'] = ref_heatmap['dataset_label'] + '\n' + ref_heatmap['metric']
    ref_table = ref_heatmap.pivot_table(index='label', columns='column', values='value', aggfunc='mean')
else:
    ref_table = pd.DataFrame()
fig, ax = plt.subplots(figsize=(max(9, ref_table.shape[1] * 1.0 if not ref_table.empty else 9), 4.8), dpi=220)
if ref_table.empty:
    ax.text(0.5, 0.5, 'No reference-baseline summary available', ha='center', va='center'); ax.axis('off')
else:
    sns.heatmap(ref_table, annot=False, cmap='viridis', linewidths=0.0, linecolor=None, ax=ax, cbar_kws={'shrink': 0.88, 'pad': 0.02})
    ax.set_title('TriShift vs reference baselines')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelrotation=58, labelsize=8)
    ax.tick_params(axis='y', labelrotation=0, labelsize=9)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)
plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3c_reference_baselines.png', bbox_inches='tight'); plt.close()

residual_plot_df = residual_summary_df[residual_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
if not residual_plot_df.empty:
    residual_plot_df['label'] = residual_plot_df['model_name'].map(MODEL_LABELS).fillna(residual_plot_df['model_name'])
render_grouped_bar(
    residual_plot_df,
    'residualized_systema_corr_20de_allpert',
    'Residualized Systema Pearson',
    OUT_ROOT / 'fig3d_residualized_systema_pearson.png',
    order=dataset_order,
)

centroid_plot_df = centroid_summary_df[centroid_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
if not centroid_plot_df.empty:
    centroid_plot_df['label'] = centroid_plot_df['model_name'].map(MODEL_LABELS).fillna(centroid_plot_df['model_name'])
render_grouped_bar(
    centroid_plot_df,
    'centroid_accuracy',
    'Centroid accuracy',
    OUT_ROOT / 'fig3e_centroid_accuracy.png',
    order=dataset_order,
)



In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.9), dpi=220)
plot_specs = [
    ('systema_corr_20de_allpert', 'Systema Pearson'),
    ('residualized_systema_corr_20de_allpert', 'Residualized Systema Pearson'),
    ('generic_projection_ratio', 'Generic-shift projection ratio'),
]
required_cols = {'model_name', 'dataset_label', 'train_distance_bin'}
if difficulty_bin_df.empty or not required_cols.issubset(difficulty_bin_df.columns):
    for ax in axes:
        ax.text(0.5, 0.5, 'No difficulty-bin summary available', ha='center', va='center')
        ax.axis('off')
else:
    from matplotlib.lines import Line2D

    work = difficulty_bin_df[difficulty_bin_df['model_name'].isin(GENERIC_SHIFT_MODELS)].copy()
    work['label'] = work['model_name'].map(MODEL_LABELS).fillna(work['model_name'])
    work['train_distance_bin'] = pd.Categorical(work['train_distance_bin'], categories=['near', 'medium', 'far'], ordered=True)
    dataset_style = {'Adamson': ('o', '-'), 'Norman (single)': ('X', '--')}
    ordered_labels = [MODEL_LABELS[m] for m in GENERIC_SHIFT_MODELS if m in set(work['model_name'])]
    for ax, (metric_col, title) in zip(axes, plot_specs):
        if metric_col not in work.columns:
            ax.text(0.5, 0.5, f'Missing {metric_col}', ha='center', va='center')
            ax.axis('off')
            continue
        for label in ordered_labels:
            for dataset_label in dataset_order:
                sub = work[(work['label'] == label) & (work['dataset_label'] == dataset_label)].copy()
                if sub.empty:
                    continue
                marker, linestyle = dataset_style.get(dataset_label, ('o', '-'))
                sns.lineplot(
                    data=sub,
                    x='train_distance_bin',
                    y=metric_col,
                    sort=False,
                    marker=marker,
                    linestyle=linestyle,
                    color=MODEL_COLORS[label],
                    linewidth=2.0,
                    markersize=7,
                    legend=False,
                    ax=ax,
                )
        ax.set_xlabel('Train-distance bin')
        ax.set_ylabel(title)
        ax.set_title(title)
        _style_axis(ax)
    model_handles = [Line2D([0], [0], color=MODEL_COLORS[label], lw=2.2, label=label) for label in ordered_labels]
    dataset_handles = [
        Line2D([0], [0], color='#444444', lw=1.8, marker='o', linestyle='-', label='Adamson'),
        Line2D([0], [0], color='#444444', lw=1.8, marker='X', linestyle='--', label='Norman (single)'),
    ]
    leg1 = axes[0].legend(handles=model_handles, title='', frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.22), ncol=min(3, len(model_handles)), borderaxespad=0.0)
    axes[0].add_artist(leg1)
    axes[-1].legend(handles=dataset_handles, title='', frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.22), ncol=2, borderaxespad=0.0)
    fig.text(0.5, 0.01, 'Note: Perturbed mean and Matching mean overlap on Adamson.', ha='center', va='bottom', fontsize=9, color='#555555')
fig.tight_layout(rect=[0.0, 0.04, 1.0, 0.89]); plt.savefig(OUT_ROOT / 'fig3g_generic_shift_dependence.png', bbox_inches='tight'); plt.close()
display(difficulty_bin_df.head())



,dataset,model_name,train_distance_bin,systema_corr_20de_allpert,residualized_systema_corr_20de_allpert,generic_projection_ratio,dataset_label
0,adamson,systema_matching_mean,far,0.100528,-0.022807,0.999278,Adamson
1,adamson,systema_matching_mean,medium,0.496597,0.367804,0.999321,Adamson
2,adamson,systema_matching_mean,near,0.428446,0.301035,0.999319,Adamson
3,adamson,systema_nonctl_mean,far,0.100528,-0.022807,0.999278,Adamson
4,adamson,systema_nonctl_mean,medium,0.496597,0.367804,0.999321,Adamson


In [7]:
case_meta, case_df = pick_reference_case()
if case_meta is not None and case_df is not None:
    case_df.to_csv(OUT_ROOT / 'fig3f_reference_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(12.8, 5.1), dpi=220)
    ax = sns.barplot(data=case_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('white'); patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(f"Reference-sensitive case: {case_meta['condition']} (split {case_meta['split_id']})")
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=min(3, case_df['Group'].nunique()))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8)
    _style_axis(ax)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig3f_reference_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_df.head(20))
else:
    print('No reference-sensitive case could be selected from current results.')
print(OUT_ROOT)



{'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4, 'score_gain': nan, 'override': True}


,Gene,Expression,Group
0,NEAT1,0.821027,Truth
1,MALAT1,0.722918,Truth
2,CSF3R,0.421455,Truth
3,PHF19,-0.357565,Truth
4,APOC1,-0.354939,Truth
5,SRM,-0.318603,Truth
6,NME1,-0.297772,Truth
7,PTMA,-0.286423,Truth
8,FABP5,-0.272752,Truth
9,RANBP1,-0.259043,Truth


E:\CODE\trishift\artifacts\paper_figures\main\Fig3_ReferenceConditioning
